In [36]:
import warnings
warnings.filterwarnings(action='ignore')
import requests
import numpy as np
import pandas as pd
from pandas import json_normalize
import folium

따릉이 서버에 post 방식으로 요청해서 서버가 응답하는 따릉이 스테이션 데이터를 받는다.

In [5]:
targetSite = 'https://www.bikeseoul.com/app/station/getStationRealtimeStatus.do'
response = requests.post(targetSite, data={
    'stationGrpSeq': 'ALL'
})
# print(response)
# print(type(response.text))
# print(response.text)

텍스트로된 따릉이 스테이션 데이터를 파이썬에서 처리 가능한 자료형으로 변환환다.

In [7]:
bicycle = response.json()
print(type(bicycle))
print(bicycle.keys())

<class 'dict'>
dict_keys(['stationImgPath', 'appUserSessionVO', 'loginYn', 'realtimeList', 'sessionId', 'stationVO', 'checkResult'])


딕셔너리 타입으로 변환환 데이터에서 따릉이 스테이션 정보가 할당된 realtimeList key의 value를 판다스 데이터프레임으로 만든다.

In [12]:
print(len(bicycle['realtimeList']))
print(type(bicycle['realtimeList']))
bicycle['realtimeList'][0]

2750
<class 'list'>


{'stationImgFileName': '',
 'stationId': 'ST-4',
 'stationName': '102. 망원역 1번출구 앞',
 'stationLongitude': '126.91062927',
 'stationLatitude': '37.55564880',
 'rackTotCnt': '15',
 'parkingBikeTotCnt': '0',
 'parkingQRBikeCnt': '0',
 'parkingELECBikeCnt': '0',
 'stationSeCd': 'RAK_002',
 'mode': None}

In [21]:
bicycle_df = json_normalize(bicycle, 'realtimeList')
bicycle_df

,stationImgFileName,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,stationSeCd,mode
0,,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,0,0,RAK_002,None
1,,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,1,0,RAK_002,None
2,,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,15,0,RAK_002,None
3,,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,17,1,RAK_002,None
4,,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,2,1,RAK_002,None
...,...,...,...,...,...,...,...,...,...,...,...
2745,,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,21,2,RAK_002,None
2746,,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,7,0,RAK_002,None
2747,,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.82738495,37.56614685,10,0,5,1,RAK_002,None
2748,,ST-3437,6191.가양역5번출구,126.85717010,37.56021500,10,0,10,2,RAK_002,None


In [22]:
bicycle_df.columns

Index(['stationImgFileName', 'stationId', 'stationName', 'stationLongitude',
       'stationLatitude', 'rackTotCnt', 'parkingBikeTotCnt',
       'parkingQRBikeCnt', 'parkingELECBikeCnt', 'stationSeCd', 'mode'],
      dtype='str')

bicycle_df 데이터프레임에서 drop() 메소드로 불필요한 컬럼을 제거한다.

stationId: 대여소 ID  
stationName: 대여소 이름  
stationLongitude: 대여소 경도  
stationLatitude: 대여소 위도  
rackTotCnt: 대여소에 주차 가능한 자전거 대수  
parkingBikeTotCnt: 대여소에 주차된 LCD형 따릉이 대수  
parkingQRBikeCnt: 대여소에 주차된 QR형 따릉이 대수  
parkingELECBikeCnt: 대여소에 주차된 새싹 따릉이 대수

In [23]:
bicycle_df[
    ['stationId', 'stationName', 'stationLongitude', 'stationLatitude', 'rackTotCnt', 'parkingBikeTotCnt', 'parkingQRBikeCnt', 'parkingELECBikeCnt']
]

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt
0,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,0,0
1,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,1,0
2,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,15,0
3,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,17,1
4,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,2,1
...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,21,2
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,7,0
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.82738495,37.56614685,10,0,5,1
2748,ST-3437,6191.가양역5번출구,126.85717010,37.56021500,10,0,10,2


In [24]:
# bicycle_df = bicycle_df.drop(columns=['stationImgFileName', 'stationSeCd', 'mode'])
# drop() 메소드에 inplace=True를 지정하면 메소드 실행 결과를 별도로 저장하지 않아도 자동으로 저장된다.
bicycle_df.drop(columns=['stationImgFileName', 'stationSeCd', 'mode'], inplace=True)
bicycle_df

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt
0,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,0,0
1,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,1,0
2,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,15,0
3,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,17,1
4,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,2,1
...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,21,2
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,7,0
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.82738495,37.56614685,10,0,5,1
2748,ST-3437,6191.가양역5번출구,126.85717010,37.56021500,10,0,10,2


데이터프레임 열이름을 변경하려면 columns 속성에 리스트 형태로 새 열이름을 넣어주면 된다.

In [27]:
bicycle_df.columns = ['id', 'name', 'lon', 'lat', 'totcnt', 'lcd', 'qr', 'elec']

경도, 위도, 자전거 주차 대수의 데이터 타입을 변경한다.

In [28]:
# bicycle_df.dtypes
bicycle_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2750 entries, 0 to 2749
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2750 non-null   str  
 1   name    2750 non-null   str  
 2   lon     2750 non-null   str  
 3   lat     2750 non-null   str  
 4   totcnt  2750 non-null   str  
 5   lcd     2750 non-null   str  
 6   qr      2750 non-null   str  
 7   elec    2750 non-null   str  
dtypes: str(8)
memory usage: 172.0 KB


In [29]:
bicycle_df['lon'] = bicycle_df['lon'].astype(float)
bicycle_df['lat'] = bicycle_df['lat'].astype(float)
bicycle_df['totcnt'] = bicycle_df['totcnt'].astype(int)
bicycle_df['lcd'] = bicycle_df['lcd'].astype(int)
bicycle_df['qr'] = bicycle_df['qr'].astype(int)
bicycle_df['elec'] = bicycle_df['elec'].astype(int)

In [30]:
bicycle_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2750 entries, 0 to 2749
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      2750 non-null   str    
 1   name    2750 non-null   str    
 2   lon     2750 non-null   float64
 3   lat     2750 non-null   float64
 4   totcnt  2750 non-null   int64  
 5   lcd     2750 non-null   int64  
 6   qr      2750 non-null   int64  
 7   elec    2750 non-null   int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 172.0 KB


따릉이 스테이션에 주차된 전체 자전거 대수를 기억하는 파생 변수를 만든다.

In [31]:
bicycle_df['total'] = bicycle_df['lcd'] + bicycle_df['qr'] + bicycle_df['elec']
bicycle_df

,id,name,lon,lat,totcnt,lcd,qr,elec,total
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,0,0,0
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,1,0,1
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,15,0,15
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,17,1,18
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,2,1,3
...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,21,2,23
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,7,0,7
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,5,1,6
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12


In [32]:
bicycle_df.head()

,id,name,lon,lat,totcnt,lcd,qr,elec,total
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,0,0,0
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,1,0,1
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,15,0,15
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,17,1,18
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,2,1,3


In [33]:
bicycle_df.tail()

,id,name,lon,lat,totcnt,lcd,qr,elec,total
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,21,2,23
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,7,0,7
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,5,1,6
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12
2749,ST-3440,6192.롯데몰김포,126.805962,37.564384,12,0,2,2,4


In [34]:
bicycle_df.to_csv('./data/bicycle.csv', index=False, encoding='utf-8-sig')

In [35]:
df = pd.read_csv('./data/bicycle.csv')
df

,id,name,lon,lat,totcnt,lcd,qr,elec,total
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,0,0,0
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,1,0,1
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,15,0,15
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,17,1,18
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,2,1,3
...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,21,2,23
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,7,0,7
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,5,1,6
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12


서울시 전체 따릉이 스테이션 시각화

In [42]:
foliumMap = folium.Map(location=[bicycle_df['lat'].mean(), bicycle_df['lon'].mean()], zoom_start=12)
for _, bicycle in bicycle_df.iterrows():
    popup = folium.Popup(f'스테이션 이름: {bicycle["name"].split(".")[1]}<br>주차된 자전거 수: {bicycle["total"]}대', max_width=300)
    folium.Marker(location=[bicycle['lat'], bicycle['lon']], popup=popup).add_to(foliumMap)
foliumMap

In [44]:
# !pip install geopy

In [45]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderRateLimited

In [46]:
geocoder = Nominatim(user_agent='mkvryu', timeout=None)

In [51]:
print(geocoder.geocode('서울시 강남구 역삼동'))

역삼동, 역삼2동, 강남구, 서울특별시, 06220, 대한민국


In [49]:
print(geocoder.reverse('37.555649,126.910629'))

망원역 1번출구 앞, 월드컵로, 서교동, 마포구, 서울특별시, 03996, 대한민국
